# 04 — Trace Analysis

Analyze execution traces: synthetic vs real, path coverage,
intent diversity, and learning progression over time.

**Requires:** `vault-exec init` + some `vault-exec run` executions

In [1]:
import { openVaultStore } from "../src/db/index.ts";

const VAULT_PATH = "../demo-vault";
const DB_PATH = `${VAULT_PATH}/.vault-exec/vault.kv`;

const db = await openVaultStore(DB_PATH);
const traces = await db.getAllTraces();
const allNotes = await db.getAllNotes();
const noteNames = new Set(allNotes.map(n => n.name));

console.log(`Total traces: ${traces.length}`);
console.log(`Notes in vault: ${noteNames.size}`);


Total traces: 137


Notes in vault: 124


In [2]:
// Trace overview
const synthetic = traces.filter(t => t.synthetic);
const real = traces.filter(t => !t.synthetic);
const withIntent = traces.filter(t => t.intent);
const withIntentEmb = traces.filter(t => t.intentEmbedding && t.intentEmbedding.length > 0);

console.log('\n=== Trace Overview ===');
console.log(`Synthetic:          ${synthetic.length}`);
console.log(`Real:               ${real.length}`);
console.log(`With intent text:   ${withIntent.length}`);
console.log(`With intent embed:  ${withIntentEmb.length}`);

// Unique targets covered
const synTargets = new Set(synthetic.map(t => t.targetNote));
const realTargets = new Set(real.map(t => t.targetNote));
const allTargets = new Set(traces.map(t => t.targetNote));

console.log(`\nTarget coverage:`);
console.log(`  Synthetic targets: ${synTargets.size}/${noteNames.size}`);
console.log(`  Real targets:      ${realTargets.size}/${noteNames.size}`);
console.log(`  Total coverage:    ${allTargets.size}/${noteNames.size}`);

// Uncovered notes
const uncovered = [...noteNames].filter(n => !allTargets.has(n));
if (uncovered.length > 0) {
  console.log(`\n⚠ Uncovered notes (no trace targets them):`);
  uncovered.forEach(n => console.log(`  - ${n}`));
}


=== Trace Overview ===


Synthetic:          98


Real:               39


With intent text:   126


With intent embed:  28


...

  - REV - Usage Snapshots


  - SHARED - Segment Owner Routing


  - SHARED - Vault Workspace README


  - Tickets


  - Usage Snapshots


In [3]:
// Trace type breakdown
const traceTypeData = [
  { type: 'Synthetic', count: synthetic.length },
  { type: 'Real (with intent)', count: real.filter(t => t.intent).length },
  { type: 'Real (no intent)', count: real.filter(t => !t.intent).length },
];

const typeSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Trace Types",
  "width": 300,
  "height": 200,
  "data": { "values": traceTypeData },
  "mark": { "type": "arc", "innerRadius": 50 },
  "encoding": {
    "theta": { "field": "count", "type": "quantitative" },
    "color": { "field": "type", "type": "nominal", "scale": { "range": ["#90CAF9", "#4CAF50", "#FF9800"] } },
    "tooltip": [{ "field": "type" }, { "field": "count" }]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": typeSpec }, { raw: true });

In [4]:
// Path analysis: execution paths and their frequency
const pathCounts = new Map<string, number>();
for (const t of traces) {
  const pathStr = t.path.join(' → ');
  pathCounts.set(pathStr, (pathCounts.get(pathStr) ?? 0) + 1);
}

console.log('\n=== Execution Paths ===');
[...pathCounts.entries()]
  .sort((a, b) => b[1] - a[1])
  .forEach(([path, count]) => {
    console.log(`  ${count}x  ${path}`);
  });

// Path length distribution
const pathLengths = traces.map(t => ({ length: t.path.length, synthetic: t.synthetic }));
const avgLen = pathLengths.reduce((s, p) => s + p.length, 0) / pathLengths.length;
console.log(`\nAvg path length: ${avgLen.toFixed(1)}`);
console.log(`Max path length: ${Math.max(...pathLengths.map(p => p.length))}`);


=== Execution Paths ===


  3x  Accounts → Contracts → Days To Renewal → NPS Records → Tickets → High Severity Ticket Count → Usage Snapshots → Account Health Inputs → Revenue Risk Score → Upsell Candidate Queue


  3x  Accounts → CSM Owners → Contracts → Account Contract Links → Contract Status Filter → Days To Renewal → Invoices → NPS Records → NPS Freshness → Overdue Invoice Count → Owner Join → Renewal Window Band → Tickets → High Severity Ticket Count → Open Ticket Aging Score → Usage Snapshots → Account Health Inputs → Revenue Risk Score → Churn Pressure Score → Executive Risk Brief → Expansion Readiness Score → Leadership Brief Draft → Owner Priority Matrix → Owner Capacity → Owner Workload Balance → Renewal Action Queue → CSM Daily Plan → Executive Escalation Payload → Executive Escalation Ack → Renewal Meeting Payload → Renewal Outreach Drafts → Risk Delta → Task Upsert Payload → Task Reassign Payload → Upsell Candidate Queue → Expansion Brief Payload → Usage Freshness → Data Completeness Score → Action Eligibility → Outreach Variant Drafts → Task Close Payload → Action Dispatch Queue → Dispatch Batch → Idempotency Key Builder → Payload Validator → Dispatch Dry Run


  3x  CRM - Contacts → SHARED - Segment Owner Routing → CRM - Inbox-to-CRM


  2x  Accounts → Contracts → Days To Renewal → Invoices → NPS Records → Overdue Invoice Count → Tickets → High Severity Ticket Count → Open Ticket Aging Score → Usage Snapshots → Account Health Inputs → Revenue Risk Score → Churn Pressure Score → Expansion Readiness Score


...

  1x  REV - Accounts → REV - CSM Owners → REV - Contracts → REV - Account Contract Links → REV - Contract Status Filter → REV - Days To Renewal → REV - Invoices → REV - NPS Records → REV - NPS Freshness → REV - Overdue Invoice Count → REV - Owner Join → REV - Renewal Window Band → REV - Tickets → REV - High Severity Ticket Count → REV - Open Ticket Aging Score → REV - Usage Snapshots → REV - Account Health Inputs → REV - Revenue Risk Score → REV - Churn Pressure Score → REV - Expansion Readiness Score → REV - Owner Priority Matrix → REV - Renewal Action Queue → REV - CSM Daily Plan → REV - Risk Delta → REV - Task Upsert Payload → REV - Usage Freshness → REV - Data Completeness Score → REV - Action Eligibility → REV - Task Close Payload


  1x  REV - Accounts → REV - CSM Owners → REV - Contracts → REV - Account Contract Links → REV - Days To Renewal → REV - Invoices → REV - NPS Records → REV - Overdue Invoice Count → REV - Owner Join → REV - Tickets → REV - High Severity Ticket Count → REV - Open Ticket Aging Score → REV - Usage Snapshots → REV - Account Health Inputs → REV - Revenue Risk Score → REV - Churn Pressure Score → REV - Expansion Readiness Score → REV - Owner Priority Matrix → REV - Owner Capacity → REV - Owner Workload Balance → REV - Renewal Action Queue → REV - CSM Daily Plan → REV - Task Upsert Payload → REV - Task Reassign Payload


  1x  REV - Accounts → REV - CSM Owners → REV - Contracts → REV - Account Contract Links → REV - Days To Renewal → REV - Invoices → REV - NPS Records → REV - Overdue Invoice Count → REV - Owner Join → REV - Tickets → REV - High Severity Ticket Count → REV - Open Ticket Aging Score → REV - Usage Snapshots → REV - Account Health Inputs → REV - Revenue Risk Score → REV - Churn Pressure Score → REV - Expansion Readiness Score → REV - Owner Priority Matrix → REV - Renewal Action Queue → REV - CSM Daily Plan → REV - Task Upsert Payload



Avg path length: 13.7


Max path length: 47


In [5]:
// Path length distribution chart
const lenCounts = new Map<number, {synthetic: number, real: number}>();
for (const t of traces) {
  const len = t.path.length;
  if (!lenCounts.has(len)) lenCounts.set(len, { synthetic: 0, real: 0 });
  if (t.synthetic) lenCounts.get(len)!.synthetic++;
  else lenCounts.get(len)!.real++;
}

const lenData: Array<{length: number, count: number, type: string}> = [];
for (const [len, counts] of lenCounts) {
  lenData.push({ length: len, count: counts.synthetic, type: 'synthetic' });
  lenData.push({ length: len, count: counts.real, type: 'real' });
}

const lenSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Path Length Distribution",
  "width": 400,
  "height": 250,
  "data": { "values": lenData },
  "mark": "bar",
  "encoding": {
    "x": { "field": "length", "type": "ordinal", "title": "Path Length" },
    "y": { "field": "count", "type": "quantitative", "title": "Traces", "stack": true },
    "color": { "field": "type", "type": "nominal" },
    "tooltip": [{ "field": "length" }, { "field": "type" }, { "field": "count" }]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": lenSpec }, { raw: true });

In [6]:
// Intent diversity: unique intents and their similarity
function cosine(a: number[], b: number[]): number {
  let dot = 0, na = 0, nb = 0;
  for (let i = 0; i < a.length; i++) {
    dot += a[i] * b[i];
    na += a[i] * a[i];
    nb += b[i] * b[i];
  }
  return dot / (Math.sqrt(na) * Math.sqrt(nb) + 1e-9);
}

const intents = withIntentEmb.map(t => ({
  intent: t.intent ?? '(none)',
  target: t.targetNote,
  embedding: t.intentEmbedding!,
}));

if (intents.length >= 2) {
  console.log('\n=== Intent Embeddings ===');
  console.log(`${intents.length} intents with embeddings\n`);
  
  // Pairwise similarity between intents
  const intentSimData: Array<{intentA: string, intentB: string, similarity: number}> = [];
  for (let i = 0; i < intents.length; i++) {
    for (let j = 0; j < intents.length; j++) {
      intentSimData.push({
        intentA: intents[i].intent.slice(0, 30),
        intentB: intents[j].intent.slice(0, 30),
        similarity: cosine(intents[i].embedding, intents[j].embedding),
      });
    }
  }
  
  const intentSpec = {
    "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
    "title": "Intent Similarity Matrix",
    "width": 400,
    "height": 400,
    "data": { "values": intentSimData },
    "mark": "rect",
    "encoding": {
      "x": { "field": "intentA", "type": "nominal", "title": null },
      "y": { "field": "intentB", "type": "nominal", "title": null },
      "color": { "field": "similarity", "type": "quantitative", "scale": { "scheme": "viridis" } },
      "tooltip": [{ "field": "intentA" }, { "field": "intentB" }, { "field": "similarity", "format": ".3f" }]
    }
  };
  await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": intentSpec }, { raw: true });
} else {
  console.log('\nNot enough intents with embeddings for similarity analysis.');
  console.log('Run more --intent queries to accumulate data.');
}


=== Intent Embeddings ===


28 intents with embeddings



In [7]:
// Trace timeline (if executedAt available)
const withTime = traces.filter(t => t.executedAt).map(t => ({
  time: t.executedAt!,
  target: t.targetNote,
  synthetic: t.synthetic,
  pathLen: t.path.length,
}));

if (withTime.length > 0) {
  console.log('\n=== Trace Timeline ===');
  console.table(withTime.slice(-20)); // last 20
} else {
  console.log('No timestamped traces available.');
}


=== Trace Timeline ===


┌───────┬────────────────────────────┬──────────────────────────────────────┬───────────┬─────────┐
│ (idx) │ time                       │ target                               │ synthetic │ pathLen │
├───────┼────────────────────────────┼──────────────────────────────────────┼───────────┼─────────┤
│     0 │ "2026-03-05T04:45:12.975Z" │ "REV - Risk Delta"                   │ true      │      11 │
│     1 │ "2026-03-05T04:45:12.975Z" │ "REV - Revenue Risk Score"           │ true      │       9 │
│     2 │ "2026-03-05T04:45:12.976Z" │ "REV - Upsell Candidate Queue"       │ true      │      10 │
│     3 │ "2026-03-05T04:45:12.977Z" │ "REV - Dispatch Audit Log"           │ true      │      47 │
│     4 │ "2026-03-05T04:45:12.977Z" │ "REV - Action Dispatch Queue"        │ true      │      42 │
│     5 │ "2026-03-05T04:45:12.978Z" │ "REV - Executive Escalation Ack"     │ true      │      12 │
│     6 │ "2026-03-05T04:45:12.978Z" │ "REV - Dispatch Dry Run"             │ true      │      46 │


In [8]:
// Training data quality assessment
console.log('\n=== Training Data Quality ===');

// Duplicate synthetic traces (init re-run problem)
const synPaths = synthetic.map(t => t.path.join(','));
const uniqueSynPaths = new Set(synPaths);
const dupRatio = 1 - uniqueSynPaths.size / Math.max(synPaths.length, 1);
console.log(`Synthetic duplicates: ${synPaths.length - uniqueSynPaths.size}/${synPaths.length} (${(dupRatio * 100).toFixed(0)}%)`);
if (dupRatio > 0.3) {
  console.log('⚠ High duplicate rate — init was likely run multiple times without clearing traces');
}

// Intent embedding coverage
const intentCoverage = withIntentEmb.length / Math.max(real.length, 1);
console.log(`\nReal traces with intent embeddings: ${withIntentEmb.length}/${real.length} (${(intentCoverage * 100).toFixed(0)}%)`);

// Notes never targeted
console.log(`\nNotes never targeted: ${uncovered.length}/${noteNames.size}`);
if (uncovered.length > 0) {
  console.log('  These notes have no training signal — GRU cannot learn to route to them.');
}


=== Training Data Quality ===


Synthetic duplicates: 0/98 (0%)



Real traces with intent embeddings: 28/39 (72%)



Notes never targeted: 26/124


  These notes have no training signal — GRU cannot learn to route to them.


In [9]:
db.close();
console.log('Done');

Done
